# 28 · Auditoria dos sorteios: o resultado depende do acaso da partição?

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Por que este passo.** A validação do artigo reparte os trechos inteiros entre pátios em cinco grupos, ao acaso, 30 vezes (sementes 42 a 71). O sorteio decide só quais trechos ficam fora do ajuste em cada rodada; não escolhe resultado. Este notebook confere, com os dados, quatro coisas: (i) a partição está correta (nenhum trecho dividido, cada quilômetro avaliado uma vez por sorteio); (ii) 30 sorteios bastam (a média não muda com mais sorteios); (iii) o resultado não é obra do acaso (teste de permutação de Ojala e Garriga, 2010, e um nulo que preserva o agrupamento espacial dos descarrilamentos); (iv) o bloco espacial tem a ordem de grandeza do alcance da dependência entre quilômetros vizinhos (Roberts et al., 2017). A repetição da partição responde à baixa replicabilidade de uma partição única (Bouckaert e Frank, 2004).

**Gates, declarados antes do código:** 0 trechos divididos; 20.936 células avaliadas por sorteio; pelo menos 100 km atingidos em cada grupo; regra do EAR 0,588; logística com atributos 0,759 (notebook 24); média da AUC com rótulos embaralhados entre 0,45 e 0,55.

In [ ]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib; matplotlib.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
import matplotlib.pyplot as plt
SEED = 42
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv", low_memory=False)
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]; gate("descarrilamentos por via", len(EV), 2681, source="notebook 11")
N = len(C); idx = pd.Series(np.arange(N), index=C["cel_id"].values); ANOS = np.arange(2011, 2026)
def matriz(ev):
    M_ = np.zeros((15, N))
    for k in range(15):
        s = ev[ev["ano"] == 2011 + k].groupby("cel_id").size(); M_[k, idx[s.index].values] = s.values
    return M_
CV_, CA_ = matriz(EV), matriz(E)
def conta(M_, a0, a1): return M_[(ANOS >= a0) & (ANOS <= a1)].sum(0)
EARN = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)] + ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)]
NUM = EARN + ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(num, cat):
    X = C[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    X = pd.concat([X, pd.get_dummies(C[cat], dtype=float)], axis=1) if cat else X; return StandardScaler().fit_transform(X.values.astype(float))
X0, XE = design(NUM, CAT), design(EARN, [])
FERC = pd.factorize(C["ferrovia"])[0]; LINC = pd.factorize(C["ferrovia"].astype(str) + "|" + C["linha"].astype(str))[0]; SEGC = pd.factorize(C["seg_id"].astype(str))[0]; KM = ((C["km_a"] + C["km_b"]) / 2).values; LN = C["len_km"].values
ri, ci = [], []
for l in range(LINC.max() + 1):
    i = np.where(LINC == l)[0]; k = KM[i]; M_ = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M_); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(N, N)); nviz = np.asarray(W.sum(1)).ravel()
def historico(a0, a1):
    n = a1 - a0 + 1; hv = conta(CV_, a0, a1); ha = conta(CA_, a0, a1); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEGC)["b"].transform("mean").values, d.groupby(LINC)["b"].transform("mean").values, d.groupby(FERC)["b"].transform("mean").values, d.groupby(FERC)["hv"].transform("mean").values / n])
dp = C["dmin_patio"].fillna(1e9).values; rg = C["rmin_geo"].fillna(1e6).values
B2 = 0.5 * np.select([dp <= 300, dp <= 600, dp <= 1200, dp <= 1800], [4, 3, 2, 1], 0) / 4 + 0.5 * np.select([rg < 500, rg < 1200, rg < 2300], [3, 2, 1], 0) / 3
def auc(y, s):
    n1 = y.sum(); n0 = len(y) - n1; return float((rankdata(s)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def auc_dentro(s, y):
    a, w = [], []
    for f in range(FERC.max() + 1):
        i = FERC == f
        if 5 <= y[i].sum() < i.sum(): a.append(auc(y[i], s[i])); w.append(y[i].sum())
    return float(np.average(a, weights=w))
def lr(Xtr, ytr, Xte): return LogisticRegression(C=0.5, max_iter=3000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
DES = np.random.default_rng(SEED).random(N)
DES = np.random.default_rng(SEED).random(N)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import average_precision_score
log_ia("28.0 dados e funções", "Claude", "FOCUS: mesmas funções, matriz e protocolo dos notebooks 24 e 25", "design, auc, lr", "gates de contagem")

# 0. Resposta e protocolo: iguais ao notebook 23 e à dissertação (trechos inteiros entre pátios fora do ajuste, 5 grupos, 30 sorteios)
n15 = conta(CV_, 2011, 2025); y = (n15 > 0).astype(int); gate("quilômetros com descarrilamento por via em 2011 a 2025", int(y.sum()), 2123, source="notebook 11")
gate("regra do EAR no período inteiro", round(auc(y, B2), 3), 0.588, tol=0.002, source="notebook 11 e dissertação v4")
KF, R = 5, 30
def folds_por(grupo, seed, k=KF):
    u = np.unique(grupo); f = dict(zip(u, np.random.default_rng(seed).permutation(len(u)) % k)); return np.array([f[g] for g in grupo])
def auc_grupo(s, yy, G, minimo=5):
    a, w = [], []
    for g in np.unique(G):
        i = G == g
        if minimo <= yy[i].sum() < i.sum(): a.append(auc(yy[i], s[i])); w.append(yy[i].sum())
    return float(np.average(a, weights=w))
def topo(s, yy):
    o = np.lexsort((DES, -s)); top = np.zeros(N, bool); top[o[(np.cumsum(LN[o]) / LN.sum()) <= 0.20]] = True; return top
def matriz_conf(s, yy):
    t = topo(s, yy); tp = int((t & (yy == 1)).sum()); fp = int((t & (yy == 0)).sum()); fn = int((~t & (yy == 1)).sum()); tn = int((~t & (yy == 0)).sum())
    assert tp + fn == yy.sum() and tp + fp + fn + tn == N
    rec = tp / (tp + fn); pre = tp / (tp + fp); return dict(VP=tp, FP=fp, FN=fn, VN=tn, recall=rec, precisao=pre, especificidade=tn / (tn + fp), F1=2 * pre * rec / (pre + rec))




In [ ]:
# 28 · Auditoria dos sorteios: os resultados dependem do acaso da partição?
# FOCUS (declarado antes do código)
#  F: auditoria metodológica do protocolo de validação do artigo (trechos inteiros entre pátios fora do ajuste, 5 grupos, 30 sorteios, sementes 42 a 71).
#  O: mostrar, com dados, que (i) a partição é correta, (ii) 30 sorteios bastam, (iii) o resultado não é obra do acaso, (iv) o bloco espacial é maior que o alcance da dependência espacial.
#  C: matriz de células e eventos do notebook 11; funções do notebook 27 (folds_por, auc, auc_grupo, lr, X0, B2).
#  U: só dados públicos; mesmas sementes; nada é ajustado para atingir alvo.
#  Gates (respostas conhecidas e tolerâncias, com fonte):
#   G1 cada trecho entre pátios cai em um único grupo em cada sorteio: 30 de 30 (definição do protocolo)
#   G2 cada célula recebe exatamente uma nota fora do ajuste por sorteio: 20.936 (notebook 11)
#   G3 todo grupo tem quilômetros com descarrilamento: mínimo de pelo menos 100 (2.123 / 5 = 425 em média)
#   G4 regra do EAR (FICO) = 0,588 (notebook 24 e dissertação), tol 0,002
#   G5 logística com atributos, média de 30 sorteios, recalculada aqui = 0,759 (tabela 24_modelos_resumo), tol 0,002
#   G6 teste de permutação: média da AUC com rótulos embaralhados entre 0,45 e 0,55 (acaso = 0,50)
import time
t0 = time.time()

# 1. Integridade da partição em cada sorteio
LINHAS = []
for r in range(R):
    fo = folds_por(SEGC, SEED + r)
    seg_fold = pd.DataFrame({"seg": SEGC, "f": fo}).groupby("seg")["f"].nunique()
    por_grupo = pd.DataFrame({"f": fo, "y": y, "km": LN}).groupby("f").agg(celulas=("y", "size"), atingidas=("y", "sum"), km=("km", "sum"))
    LINHAS.append({"sorteio": r + 1, "semente": SEED + r, "trechos_divididos": int((seg_fold > 1).sum()), "celulas_avaliadas": int(len(fo)),
                   "min_celulas_grupo": int(por_grupo["celulas"].min()), "max_celulas_grupo": int(por_grupo["celulas"].max()),
                   "min_atingidas_grupo": int(por_grupo["atingidas"].min()), "max_atingidas_grupo": int(por_grupo["atingidas"].max()),
                   "taxa_min_grupo": float((por_grupo["atingidas"] / por_grupo["celulas"]).min()), "taxa_max_grupo": float((por_grupo["atingidas"] / por_grupo["celulas"]).max())})
P = pd.DataFrame(LINHAS)
gate("G1 trechos divididos entre grupos (todos os sorteios)", int(P["trechos_divididos"].sum()), 0, source="definição do protocolo")
gate("G2 células avaliadas por sorteio", int(P["celulas_avaliadas"].min()), N, source="notebook 11")
gate_between("G3 mínimo de km atingidos em um grupo", int(P["min_atingidas_grupo"].min()), 100, 2123, source="2.123 / 5 = 425 em média")
P.to_csv(f"{TAB}/28_particoes_por_sorteio.csv", index=False)

# 2. Logística com atributos recalculada nos 30 sorteios (rota independente da tabela do notebook 24)
gate("G4 regra do EAR (FICO)", round(auc(y, B2), 3), 0.588, tol=0.002, source="notebook 24 e dissertação v4")
AUCS = []
for r in range(R):
    fo = folds_por(SEGC, SEED + r); s = np.zeros(N)
    for k in range(KF):
        te = fo == k; s[te] = lr(X0[~te], y[~te], X0[te])
    AUCS.append({"sorteio": r + 1, "auc_pais": auc(y, s), "auc_malhas": auc_grupo(s, y, FERC)})
A = pd.DataFrame(AUCS)
gate("G5 logística com atributos, média de 30 sorteios", round(A["auc_pais"].mean(), 3), 0.759, tol=0.002, source="24_modelos_resumo.csv")
# estabilidade de Monte Carlo: média acumulada e erro-padrão da média
A["media_acumulada"] = A["auc_pais"].expanding().mean(); A["ep_media"] = A["auc_pais"].expanding().std() / np.sqrt(A["sorteio"])
A.to_csv(f"{TAB}/28_logistica_por_sorteio.csv", index=False)
EST = {"media_30": A["auc_pais"].mean(), "dp_entre_sorteios": A["auc_pais"].std(), "min": A["auc_pais"].min(), "max": A["auc_pais"].max(),
       "ep_media_30": A["auc_pais"].std() / np.sqrt(R), "media_10": A["auc_pais"].iloc[:10].mean(), "media_20": A["auc_pais"].iloc[:20].mean(),
       "malhas_media": A["auc_malhas"].mean(), "malhas_dp": A["auc_malhas"].std()}
print("estabilidade:", {k: round(v, 4) for k, v in EST.items()}, round(time.time() - t0), "s"); sys.stdout.flush()

# 3. Teste de permutação (rótulos embaralhados entre células), mesma partição do sorteio 1
B = 100; fo = folds_por(SEGC, SEED); rng = np.random.default_rng(2026); NUL = []
obs = A.loc[0, "auc_pais"]; obs_m = A.loc[0, "auc_malhas"]
for b in range(B):
    yp = rng.permutation(y); s = np.zeros(N)
    for k in range(KF):
        te = fo == k; s[te] = lr(X0[~te], yp[~te], X0[te])
    NUL.append({"perm": b + 1, "auc_pais": auc(yp, s), "auc_malhas": auc_grupo(s, yp, FERC)})
NU = pd.DataFrame(NUL)
gate_between("G6 média da AUC com rótulos embaralhados", round(NU["auc_pais"].mean(), 3), 0.45, 0.55, source="acaso = 0,50")
p_pais = (1 + (NU["auc_pais"] >= obs).sum()) / (1 + B); p_malhas = (1 + (NU["auc_malhas"] >= obs_m).sum()) / (1 + B)
NU.to_csv(f"{TAB}/28_permutacao_rotulos.csv", index=False)
print("permutação:", round(NU["auc_pais"].mean(), 3), round(NU["auc_pais"].max(), 3), "obs", round(obs, 3), "p", round(p_pais, 3), "| malhas", round(NU["auc_malhas"].max(), 3), "obs", round(obs_m, 3), "p", round(p_malhas, 3), round(time.time() - t0), "s"); sys.stdout.flush()

# 4. Nulo que mantém o agrupamento espacial: deslocamento circular do vetor de descarrilamentos ao longo da malha ordenada
#    (as células estão ordenadas por ferrovia, linha e km; o deslocamento preserva a distância entre quilômetros atingidos)
DESL = []; rng2 = np.random.default_rng(2027)
for b in range(B):
    sh = int(rng2.integers(500, N - 500)); yp = np.roll(y, sh); s = np.zeros(N)
    for k in range(KF):
        te = fo == k; s[te] = lr(X0[~te], yp[~te], X0[te])
    DESL.append({"perm": b + 1, "deslocamento": sh, "auc_pais": auc(yp, s), "auc_malhas": auc_grupo(s, yp, FERC)})
DS = pd.DataFrame(DESL); DS.to_csv(f"{TAB}/28_deslocamento_circular.csv", index=False)
p_desl = (1 + (DS["auc_pais"] >= obs).sum()) / (1 + B); p_desl_m = (1 + (DS["auc_malhas"] >= obs_m).sum()) / (1 + B)
print("deslocamento:", round(DS["auc_pais"].mean(), 3), round(DS["auc_pais"].max(), 3), "p", round(p_desl, 3), "| malhas", round(DS["auc_malhas"].mean(), 3), round(DS["auc_malhas"].max(), 3), "p", round(p_desl_m, 3), round(time.time() - t0), "s"); sys.stdout.flush()

# 5. Alcance da dependência espacial ao longo da linha e tamanho dos blocos
#    razão = P(km i+d atingido | km i atingido) / taxa-base da linha, para d de 1 a 40 km, na mesma linha
RAZ = []
lin_km = pd.DataFrame({"lin": LINC, "km": np.round(KM).astype(int), "y": y})
for d in range(1, 41):
    num = den = esp = 0.0
    for l, g in lin_km.groupby("lin"):
        if g["y"].sum() == 0: continue
        m = dict(zip(g["km"], g["y"])); base = g["y"].mean()
        pares = [(m[k], m[k + d]) for k in m if (k + d) in m]
        if not pares: continue
        a = np.array(pares); hit = a[:, 0] == 1
        num += a[hit, 1].sum(); den += hit.sum(); esp += hit.sum() * base
    RAZ.append({"distancia_km": d, "prob_condicional": num / den, "esperado_sem_dependencia": esp / den, "razao": (num / den) / (esp / den)})
RZ = pd.DataFrame(RAZ); RZ.to_csv(f"{TAB}/28_dependencia_espacial.csv", index=False)
alc = int(RZ.loc[RZ["razao"] < 1.10, "distancia_km"].min()) if (RZ["razao"] < 1.10).any() else 40
seg_len = pd.DataFrame({"seg": SEGC, "km": LN, "y": y}).groupby("seg").agg(km=("km", "sum"), y=("y", "sum"))
EST.update({"alcance_km_razao_menor_1_1": alc, "mediana_km_trecho": float(seg_len["km"].median()),
            "km_atingidos_em_trechos_maiores_que_alcance": float(seg_len.loc[seg_len["km"] > alc, "y"].sum() / seg_len["y"].sum()),
            "p_permutacao_pais": p_pais, "p_permutacao_malhas": p_malhas, "p_deslocamento_pais": p_desl, "p_deslocamento_malhas": p_desl_m,
            "nulo_perm_media": NU["auc_pais"].mean(), "nulo_perm_max": NU["auc_pais"].max(), "nulo_desl_media": DS["auc_pais"].mean(), "nulo_desl_max": DS["auc_pais"].max(),
            "nulo_desl_malhas_media": DS["auc_malhas"].mean(), "nulo_desl_malhas_max": DS["auc_malhas"].max(), "obs_sorteio1": obs, "obs_malhas_sorteio1": obs_m})
pd.Series(EST).to_csv(f"{TAB}/28_resumo_auditoria.csv", header=["valor"])
print(RZ.head(12).round(3).to_string()); print("alcance", alc, "mediana trecho", EST["mediana_km_trecho"], "parcela", round(EST["km_atingidos_em_trechos_maiores_que_alcance"], 3))
log_ia("28 auditoria dos sorteios", "Claude", "FOCUS: integridade da partição, estabilidade de Monte Carlo, permutação, deslocamento circular, alcance espacial", "28_*.csv", "6 gates")
print("fim", round(time.time() - t0), "s")

exportar_registros("28")

---
## Referências

Bouckaert, R. R., & Frank, E. (2004). Evaluating the replicability of significance tests for comparing learning algorithms. In *PAKDD 2004* (LNCS 3056, pp. 3–12). Springer. https://doi.org/10.1007/978-3-540-24775-3_3

Ojala, M., & Garriga, G. C. (2010). Permutation tests for studying classifier performance. *Journal of Machine Learning Research, 11*, 1833–1863.

Roberts, D. R., et al. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.